In [1]:
import pandas as pd
from pathlib import Path
import re
DATA_DIR = Path(".")

movies = pd.read_csv(DATA_DIR / "movies.csv")
belief = pd.read_csv(DATA_DIR / "belief_data.csv")
ratings = pd.read_csv(DATA_DIR / "user_rating_history.csv")

movies.head(), belief.head(), ratings.head()

(   movieId                               title  \
 0        1                    Toy Story (1995)   
 1        2                      Jumanji (1995)   
 2        3             Grumpier Old Men (1995)   
 3        4            Waiting to Exhale (1995)   
 4        5  Father of the Bride Part II (1995)   
 
                                         genres  
 0  Adventure|Animation|Children|Comedy|Fantasy  
 1                   Adventure|Children|Fantasy  
 2                               Comedy|Romance  
 3                         Comedy|Drama|Romance  
 4                                       Comedy  ,
    userId  movieId  isSeen watchDate  userElicitRating  userPredictRating  \
 0   53982        1      -1       NaN              -1.0               -1.0   
 1   56737        1      -1       NaN              -1.0               -1.0   
 2   57704        1      -1       NaN              -1.0               -1.0   
 3   58881        1      -1       NaN              -1.0               -1.0   
 

In [3]:
movies = pd.read_csv("movies.csv")
movies["genres"] = movies["genres"].fillna("(no genres listed)")
movies["genres_split"] = movies["genres"].str.split("|")
movies_exploded = movies.explode("genres_split")

genre_counts = (
    movies_exploded.groupby("genres_split")
    .agg(movie_count=("movieId", "nunique"))
    .sort_values("movie_count", ascending=False)
)

total_movies = movies["movieId"].nunique()
genre_counts["share"] = genre_counts["movie_count"] / total_movies

print(genre_counts)

                    movie_count     share
genres_split                             
Drama                     40363  0.384150
Comedy                    27140  0.258302
Thriller                  14049  0.133710
Romance                   12034  0.114532
Action                    11454  0.109012
Documentary               11453  0.109002
Horror                    10461  0.099561
(no genres listed)         9162  0.087198
Crime                      8266  0.078671
Adventure                  6358  0.060511
Sci-Fi                     5693  0.054182
Animation                  5315  0.050585
Children                   5272  0.050176
Mystery                    4711  0.044836
Fantasy                    4523  0.043047
War                        2679  0.025497
Western                    2017  0.019197
Musical                    1131  0.010764
Film-Noir                   366  0.003483
IMAX                        197  0.001875


In [2]:
romance_mask = movies["genres"].str.contains("Romance", na=False)
romance_movies = movies[romance_mask].copy()

total_movies = len(movies)
num_romance = len(romance_movies)
romance_share = num_romance / total_movies

romance_overview = pd.DataFrame({
    "metric": ["total_movies", "romance_movies", "romance_share"],
    "value": [total_movies, num_romance, romance_share]
})

print("Romance overview:")
print(romance_overview)


Romance overview:
           metric          value
0    total_movies  105071.000000
1  romance_movies   12034.000000
2   romance_share       0.114532


In [6]:
year_pattern = re.compile(r"\((\d{4})\)")

def extract_year(title: str):
    m = year_pattern.search(title)
    return int(m.group(1)) if m else None

movies["year"] = movies["title"].apply(extract_year)
movies["decade"] = (movies["year"] // 10 * 10).astype("Int64")

romance_movies = movies[romance_mask].copy()

romance_by_decade = (
    romance_movies
    .dropna(subset=["decade"])
    .groupby("decade", as_index=False)
    .agg(num_movies=("movieId", "nunique"))
    .sort_values("decade")
)

print("Romance movies by decade:")
print(romance_by_decade)


Romance movies by decade:
    decade  num_movies
0     1890           3
1     1900          11
2     1910          52
3     1920         179
4     1930         737
5     1940         607
6     1950         569
7     1960         476
8     1970         453
9     1980         700
10    1990        1222
11    2000        2446
12    2010        3193
13    2020        1347


In [ ]:
romance_ids = romance_movies["movieId"].unique()

ratings_romance = ratings[ratings["movieId"].isin(romance_ids)].copy()

print("Number of Romance rating records:", len(ratings_romance))

romance_rating_summary = ratings_romance["rating"].describe()
print("Romance rating summary:")
print(romance_rating_summary)

ratings_romance = ratings_romance.merge(
    romance_movies[["movieId", "decade"]],
    on="movieId",
    how="left"
)

ratings_romance_by_decade = (
    ratings_romance
    .dropna(subset=["decade"])
    .groupby("decade", as_index=False)
    .agg(
        num_ratings=("rating", "size"),
        avg_rating=("rating", "mean")
    )
    .sort_values("decade")
)

print("Romance rating behavior by decade:")
print(ratings_romance_by_decade)


Number of Romance rating records: 265870
Romance rating summary:
count    261595.000000
mean          2.960641
std           1.634837
min          -1.000000
25%           2.500000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64
Romance rating behavior by decade:
    decade  num_ratings  avg_rating
0     1890           12    2.041667
1     1900           34    1.308824
2     1910          139    1.602190
3     1920         1302    2.394961
4     1930         4862    2.425583
5     1940         5909    2.840515
6     1950         8034    2.939285
7     1960         7278    2.909814
8     1970         5103    2.727390
9     1980        17670    2.981642
10    1990        56885    3.081440
11    2000        97767    3.003679
12    2010        50625    2.916578
13    2020         9056    2.571308


In [ ]:
belief_romance = belief[belief["movieId"].isin(romance_ids)].copy()

pred_unseen = belief_romance[
    (belief_romance["isSeen"] == 0) &
    (belief_romance["userPredictRating"].notna())
].copy()

print("Number of Romance belief records with userPredictRating:", len(pred_unseen))

pred_vs_actual = pred_unseen.merge(
    ratings_romance[["userId", "movieId", "rating"]],
    on=["userId", "movieId"],
    how="inner",
    suffixes=("_belief", "_rating")
)

print("Number of prediction records with actual ratings:", len(pred_vs_actual))

pred_vs_actual["prediction_error"] = (
    pred_vs_actual["userPredictRating"] - pred_vs_actual["rating"]
)
pred_vs_actual["abs_error"] = pred_vs_actual["prediction_error"].abs()

prediction_error_summary = pred_vs_actual["prediction_error"].describe()
abs_error_summary = pred_vs_actual["abs_error"].describe()

print("Prediction error summary (userPredictRating - rating):")
print(prediction_error_summary)

print("\nAbsolute prediction error summary:")
print(abs_error_summary)


Number of Romance belief records with userPredictRating: 3931
Number of prediction records with actual ratings: 319
Prediction error summary (userPredictRating - rating):
count    279.000000
mean       0.718638
std        1.485739
min       -4.500000
25%        0.000000
50%        0.500000
75%        1.500000
max        6.000000
Name: prediction_error, dtype: float64

Absolute prediction error summary:
count    279.000000
mean       1.073477
std        1.252686
min        0.000000
25%        0.000000
50%        0.500000
75%        1.500000
max        6.000000
Name: abs_error, dtype: float64


In [ ]:
movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "user_rating_history.csv")

print(movies.head())
print(ratings.head())

valid_ratings_mask = ratings["rating"].between(0, 5)
ratings = ratings[valid_ratings_mask].copy()

movies["genres"] = movies["genres"].fillna("(no genres listed)")

romance_mask = movies["genres"].str.contains("Romance", na=False)
romance_movies = movies[romance_mask].copy()
romance_ids = romance_movies["movieId"].unique()

print("Total movies:", len(movies))
print("Romance movies:", len(romance_movies))

year_pattern = re.compile(r"\((\d{4})\)")

def extract_year(title: str):
    m = year_pattern.search(str(title))
    if not m:
        return None
    year = int(m.group(1))
    if 1900 <= year <= 2025:
        return year
    else:
        return None

movies["year"] = movies["title"].apply(extract_year)
movies["decade"] = (movies["year"] // 10 * 10).astype("Int64")

romance_movies = movies[romance_mask].copy()
print("Romance movies with valid year:", romance_movies["year"].notna().sum())

ratings_full = ratings.merge(
    movies[["movieId", "genres", "decade"]],
    on="movieId",
    how="left"
)

ratings_romance = ratings_full[ratings_full["genres"].str.contains("Romance", na=False)].copy()
ratings_nonromance = ratings_full[~ratings_full["genres"].str.contains("Romance", na=False)].copy()

print("\nNumber of Romance rating records:", len(ratings_romance))
print("Number of Non-Romance rating records:", len(ratings_nonromance))

print("\nRomance rating summary:")
print(ratings_romance["rating"].describe())

print("\nNon-Romance rating summary:")
print(ratings_nonromance["rating"].describe())

ratings_romance_by_decade = (
    ratings_romance
    .dropna(subset=["decade"])
    .groupby("decade", as_index=False)
    .agg(
        num_ratings=("rating", "size"),
        avg_rating=("rating", "mean")
    )
    .sort_values("decade")
)

print("\nRomance rating behavior by decade:")
print(ratings_romance_by_decade)

ratings_nonromance_by_decade = (
    ratings_nonromance
    .dropna(subset=["decade"])
    .groupby("decade", as_index=False)
    .agg(
        num_ratings=("rating", "size"),
        avg_rating=("rating", "mean")
    )
    .sort_values("decade")
)

print("\nNon-Romance rating behavior by decade:")
print(ratings_nonromance_by_decade)

romance_movie_stats = (
    ratings_romance
    .groupby("movieId", as_index=False)
    .agg(
        num_ratings=("rating", "size"),
        avg_rating=("rating", "mean")
    )
    .merge(
        romance_movies[["movieId", "title", "decade"]],
        on="movieId",
        how="left"
    )
)

print("\nRomance movie-level stats (popularity vs quality):")
print(romance_movie_stats.head())

top_popular_romance = romance_movie_stats.sort_values("num_ratings", ascending=False).head(20)
print("\nTop 20 most-rated Romance movies:")
print(top_popular_romance[["title", "num_ratings", "avg_rating"]])


top_highrated_romance = (
    romance_movie_stats[romance_movie_stats["num_ratings"] >= 100]
    .sort_values("avg_rating", ascending=False)
    .head(20)
)
print("\nTop 20 highest-rated Romance movies (num_ratings >= 100):")
print(top_highrated_romance[["title", "num_ratings", "avg_rating"]])

movies_exploded = movies.assign(
    genres_split=movies["genres"].str.split("|")
).explode("genres_split")

ratings_genre = ratings.merge(
    movies_exploded[["movieId", "genres_split"]],
    on="movieId",
    how="left"
)

genre_stats = (
    ratings_genre
    .groupby("genres_split", as_index=False)
    .agg(
        num_ratings=("rating", "size"),
        avg_rating=("rating", "mean"),
        rating_std=("rating", "std")
    )
    .sort_values("num_ratings", ascending=False)
)

print("\nGenre-level rating stats (including Romance):")
print(genre_stats.head(20))

romance_row = genre_stats[genre_stats["genres_split"] == "Romance"]
print("\nRomance genre stats:")
print(romance_row)


user_romance_stats = (
    ratings_romance
    .groupby("userId", as_index=False)
    .agg(
        num_romance_ratings=("rating", "size"),
        avg_romance_rating=("rating", "mean")
    )
)

print("\nUser-level Romance taste stats (head):")
print(user_romance_stats.head())

romance_lovers = user_romance_stats[
    (user_romance_stats["num_romance_ratings"] >= 20) &
    (user_romance_stats["avg_romance_rating"] >= 4.0)
]
romance_cynics = user_romance_stats[
    (user_romance_stats["num_romance_ratings"] >= 20) &
    (user_romance_stats["avg_romance_rating"] <= 2.5)
]

print("\nPotential 'Romance lovers' (>=20 ratings, avg>=4):", len(romance_lovers))
print("Potential 'Romance cynics' (>=20 ratings, avg<=2.5):", len(romance_cynics))

ratings_romance_by_decade.to_csv("out_romance_ratings_by_decade.csv", index=False)
ratings_nonromance_by_decade.to_csv("out_nonromance_ratings_by_decade.csv", index=False)
romance_movie_stats.to_csv("out_romance_movie_stats.csv", index=False)
genre_stats.to_csv("out_genre_rating_stats.csv", index=False)
user_romance_stats.to_csv("out_user_romance_taste.csv", index=False)

print("\nAll summary tables saved to CSV.")


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating               tstamp
0   42170        1     4.0  1998-06-18 16:31:37
1   42170        7     4.0  1998-06-18 16:31:37
2   42170       17     4.0  1998-06-18 16:31:37
3   42170       24     2.0  1997-11-07 13:41:17
4   42170       36     2.0  1997-11-07 13:27:51
Total movies: 105071
Romance movies: 12034
Romance movies with valid year: 11992

Number of Romance rating records: 2

In [ ]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("user_rating_history.csv")

movies["genres"] = movies["genres"].fillna("")
movies["is_romance"] = movies["genres"].str.contains("Romance", case=False, na=False)

movies["GenreType"] = movies["is_romance"].replace({True: "Romance", False: "Non-Romance"})

combined = ratings.merge(
    movies[["movieId", "GenreType"]],
    on="movieId",
    how="left"
)

combined = combined[["userId", "movieId", "rating", "GenreType"]]

combined.to_csv("combined_ratings.csv", index=False)

print("Saved: combined_ratings.csv, with shape:", combined.shape)
print(combined.head())


Saved: combined_ratings.csv, with shape: (2046124, 4)
   userId  movieId  rating    GenreType
0   42170        1     4.0  Non-Romance
1   42170        7     4.0      Romance
2   42170       17     4.0      Romance
3   42170       24     2.0  Non-Romance
4   42170       36     2.0  Non-Romance
